# 🧹 Building a Reusable Data-Cleaning Pipeline
**Chapter:** Advanced Data Wrangling & Transformation

**Goal:** Build a simple, reusable pipeline that cleans any messy dataset in 4 steps.

## Step 0 — Create Sample Dirty Data

In [1]:
import pandas as pd
import numpy as np

# Messy dataset
dirty = pd.DataFrame({
    'Name':    ['  Alice ', 'BOB', ' charlie', 'Alice ', None, 'bob'],
    'Age':     [25, None, 30, 25, 40, None],
    'Salary':  ['50000', '60000', None, '50000', '70000', '60000'],
    'Email':   ['a@mail.com', 'b@mail.com', 'c@mail.com', 'a@mail.com', 'd@mail.com', 'b@mail.com']
})

print('=== DIRTY DATA ===')
dirty

=== DIRTY DATA ===


,Name,Age,Salary,Email
0,Alice,25.0,50000,a@mail.com
1,BOB,NaN,60000,b@mail.com
2,charlie,30.0,None,c@mail.com
3,Alice,25.0,50000,a@mail.com
4,None,40.0,70000,d@mail.com
5,bob,NaN,60000,b@mail.com


---
## Step 1 — Define Individual Cleaning Functions
Each function does **one job only** (Single Responsibility Principle).

In [2]:
def strip_and_lower(df, columns):
    """Strip whitespace & lowercase text columns."""
    df = df.copy()
    for col in columns:
        df[col] = df[col].astype(str).str.strip().str.lower()
        df[col] = df[col].replace('none', np.nan)  # fix stringified None
    return df


def fix_dtypes(df, dtype_map):
    """Convert columns to correct data types."""
    df = df.copy()
    for col, dtype in dtype_map.items():
        df[col] = pd.to_numeric(df[col], errors='coerce') if dtype == 'number' else df[col].astype(dtype)
    return df


def fill_missing(df, fill_map):
    """Fill missing values with given strategies."""
    df = df.copy()
    for col, strategy in fill_map.items():
        if strategy == 'median':
            df[col] = df[col].fillna(df[col].median())
        elif strategy == 'mode':
            df[col] = df[col].fillna(df[col].mode()[0])
        elif strategy == 'drop':
            df = df.dropna(subset=[col])
        else:
            df[col] = df[col].fillna(strategy)
    return df


def drop_duplicates(df, subset=None):
    """Remove duplicate rows."""
    return df.drop_duplicates(subset=subset).reset_index(drop=True)


print('✅ 4 cleaning functions defined!')

✅ 4 cleaning functions defined!


---
## Step 2 — Build the Pipeline Function
This chains the steps together — like a **car wash**.

In [3]:
def cleaning_pipeline(df):
    """Master pipeline — runs all cleaning steps in order."""

    # Step 1: Standardize text
    df = strip_and_lower(df, columns=['Name'])

    # Step 2: Fix data types
    df = fix_dtypes(df, dtype_map={'Salary': 'number', 'Age': 'number'})

    # Step 3: Handle missing values
    df = fill_missing(df, fill_map={
        'Name':   'drop',      # drop rows with no name
        'Age':    'median',     # fill age with median
        'Salary': 'median'      # fill salary with median
    })

    # Step 4: Remove duplicates
    df = drop_duplicates(df, subset=['Name', 'Email'])

    return df


print('✅ Pipeline ready!')

✅ Pipeline ready!


---
## Step 3 — Run the Pipeline

In [8]:
clean = cleaning_pipeline(dirty)

print('=== CLEAN DATA ===')
print(clean)

print("\n")

print('=== DIRTY DATA ===')
print(dirty)

=== CLEAN DATA ===
      Name   Age   Salary       Email
0    alice  25.0  50000.0  a@mail.com
1      bob  25.0  60000.0  b@mail.com
2  charlie  30.0  55000.0  c@mail.com


=== DIRTY DATA ===
       Name   Age Salary       Email
0    Alice   25.0  50000  a@mail.com
1       BOB   NaN  60000  b@mail.com
2   charlie  30.0   None  c@mail.com
3    Alice   25.0  50000  a@mail.com
4      None  40.0  70000  d@mail.com
5       bob   NaN  60000  b@mail.com


---
## ✅ Summary

| Step | Function | What It Does |
|------|----------|-------------|
| 1 | `strip_and_lower()` | Cleans whitespace & casing |
| 2 | `fix_dtypes()` | Converts to correct types |
| 3 | `fill_missing()` | Fills or drops nulls |
| 4 | `drop_duplicates()` | Removes duplicate rows |

**Key Takeaway:** Write small functions → chain them in a pipeline → reuse on any dataset! 🚀